# Bilateral Constriction · GPU

حل **جریان پایای دوبعدی سیال** در کانالی با دو نیم‌دایرهٔ روبه‌روی هم، مطابق تصویر نمادین شما.
**فعلاً هیچ ذره یا پارسل گاوسی مدل نمی‌شود.** ورودی/خروجی باز، دیواره‌های بدون لغزش و تقارن خط وسط اعمال می‌شوند.
سه حالت Re=10,20,50 و یک کنترل تحلیلی کانال مستقیم اجرا می‌شوند.

در Colab، GPU را از Runtime → Change runtime type فعال و سپس Run all کنید.
این notebook کد لازم را همراه خود دارد؛ clone، ZIP داده یا فایل خارجی لازم نیست.
هدف پیش‌فرض حداکثر ۵۵ دقیقه از اولین سلول، شامل حداکثر ۳۵ دقیقه آموزش است؛ برای شکل‌ها و ZIP بودجهٔ جدا رزرو می‌شود.
سقف زمان تضمین دقت یا تکمیل همهٔ خروجی‌ها روی هر سخت‌افزار نیست؛ وضعیت واقعی در گزارش باقی می‌ماند.

انیمیشن‌ها اسکن یک میدان **پایا** هستند و گذرای سیال یا حرکت ذره را نشان نمی‌دهند.
کنترل پوازی بررسی حد کانال مستقیم است؛ اعتبارسنجی مقاله‌ای هندسهٔ جدید ادعا نمی‌شود.

In [ ]:
import time, os
from pathlib import Path
from datetime import datetime, timezone
import subprocess, sys, json

NOTEBOOK_STARTED = time.monotonic()
TOTAL_MINUTES = 55       # سقف کل؛ بیشتر از 55 مجاز نیست
TRAIN_MINUTES = 35       # سهم کل آموزش؛ بین چهار حالت تقسیم می‌شود
PRECISION = 'float32'    # روی T4 مناسب‌تر؛ float64 کندتر است
OUTPUT_BASE = Path.cwd() / 'constriction_gpu_results'
# برای ماندگاری، می‌توانید OUTPUT_BASE را به مسیر Drive متصل‌شده تغییر دهید.
assert 0 < TOTAL_MINUTES <= 55 and TRAIN_MINUTES > 0
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
WORK = Path.cwd() / f'constriction_gpu_source_{RUN_ID}'
OUT = OUTPUT_BASE / f'run_{RUN_ID}'
NOTEBOOK_DEADLINE = NOTEBOOK_STARTED + TOTAL_MINUTES * 60
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
print('Output:', OUT)

# H=1: نسبت‌های هندسهٔ نمادین؛ قابل تغییر
GEOMETRY = {
    'length': 2.5,
    'center_fraction': 0.4,
    'radius': 0.25,
    'reynolds': [10.0, 20.0, 50.0],
}

## کد همراه notebook
سلول بعدی بستهٔ کد را با SHA-256 بررسی و در پوشه‌ای جدید استخراج می‌کند. فایل‌های قبلی بازنویسی نمی‌شوند.

In [ ]:
import base64, hashlib, io, zipfile
SOURCE_B64 = '...'
SOURCE_SHA256 = '404734f621f125d4b78ffa4d8f3d157c6a649ed2894c069c253c2688735c2394'
payload = base64.b64decode(SOURCE_B64)
assert hashlib.sha256(payload).hexdigest() == SOURCE_SHA256, 'Source checksum mismatch'
WORK.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert archive.testzip() is None
    for item in archive.infolist():
        assert (WORK / item.filename).resolve().is_relative_to(WORK.resolve())
    archive.extractall(WORK)
print('Source verified:', SOURCE_SHA256)

## آماده‌سازی GPU
PyTorch موجود در محیط حفظ می‌شود تا نسخهٔ CUDA با نسخهٔ CPU جایگزین نشود.
اگر GPU در دسترس نباشد، این notebook صریحاً متوقف می‌شود؛ آموزش مخفیانه روی CPU ادامه پیدا نمی‌کند.

In [ ]:
import importlib.util, importlib.metadata
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU را از تنظیمات runtime فعال کنید و دوباره Run all بزنید.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2, 2)

# Colab/Kaggle معمولاً این بسته‌ها را دارند؛ فقط وابستگی ناسازگار/غایب نصب می‌شود.
constraints = {
    'numpy': ('numpy>=1.26,<3', (1, 26), (3, 0)),
    'scipy': ('scipy>=1.12,<2', (1, 12), (2, 0)),
    'pandas': ('pandas>=2.2,<3', (2, 2), (3, 0)),
    'matplotlib': ('matplotlib>=3.8,<4', (3, 8), (4, 0)),
    'Pillow': ('Pillow>=10,<13', (10, 0), (13, 0)),
    'psutil': ('psutil>=5.9,<8', (5, 9), (8, 0)),
}
needed = []
for distribution, (requirement, lower, upper) in constraints.items():
    try:
        version = tuple(int(p) for p in importlib.metadata.version(distribution).split('.')[:2])
        if not lower <= version < upper:
            needed.append(requirement)
    except (importlib.metadata.PackageNotFoundError, ValueError):
        needed.append(requirement)
if needed:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *needed],
                   check=True, timeout=min(240, max(1, NOTEBOOK_DEADLINE-time.monotonic()-120)))
print('Dependencies ready. Elapsed:', round((time.monotonic()-NOTEBOOK_STARTED)/60, 2), 'min')

## پیش‌نمایش هندسه
پیش‌فرض: L/H=2.5، r/H=.25، x₀/L=.4؛ ارتفاع گلوگاه .5H است. این ابعاد تخمینی و قابل تغییرند.

In [ ]:
sys.path.insert(0, str(WORK))
from constricted_flow.model import Settings, Geometry
from constricted_flow.report import domain
import matplotlib.pyplot as plt
from IPython.display import display, Image
import io
geometry_settings = Settings(**GEOMETRY).validate()
geometry = Geometry(geometry_settings.length, geometry_settings.center_fraction, geometry_settings.radius)
fig, ax = plt.subplots(figsize=(10, 4))
domain(ax, geometry)
ax.set_title('Channel geometry')
preview_buffer = io.BytesIO()
fig.savefig(preview_buffer, format='png', dpi=110, bbox_inches='tight')
display(Image(data=preview_buffer.getvalue()))
plt.close(fig)
print('Minimum gap / H:', 1 - 2 * geometry.radius)

## هندسه و آموزش
ابتدا هندسه را بررسی کنید. داخل نیم‌دایره‌ها از دامنهٔ حل حذف شده است و شرط بدون لغزش روی قوس‌ها اعمال می‌شود.
آموزش چهار حالت مستقل با Adam، float32 و بودجهٔ زمانی محدود انجام می‌شود.
کنترل کانال مستقیم حداکثر ۹۰ ثانیه آموزش می‌گیرد؛ زمان باقی‌مانده بین حالت‌های گرفتگی‌دار تقسیم می‌شود.
خطای زیاد مانع رسم نتایج نیست؛ کیفیت بر اساس residual مستقل و خطای دبی در گزارش ثبت می‌شود.

In [ ]:
import signal, psutil
remaining_seconds = NOTEBOOK_DEADLINE - time.monotonic() - 20
if remaining_seconds < 180:
    raise RuntimeError('زمان کافی برای اجرای کامل باقی نمانده؛ runtime را آماده و notebook را دوباره اجرا کنید.')
config_path = WORK / 'geometry_config.json'
config_path.write_text(json.dumps(GEOMETRY, indent=2))
command = [sys.executable, '-u', str(WORK / 'run_constriction.py'),
           '--out', str(OUT), '--config', str(config_path),
           '--device', 'cuda', '--precision', PRECISION,
           '--max-minutes', str(min(55, remaining_seconds/60)),
           '--train-minutes', str(TRAIN_MINUTES)]
# نمایش زندهٔ وضعیت؛ ناظر مستقل کل پردازش را پیش از پایان بودجه متوقف می‌کند.
import threading
proc = subprocess.Popen(command, cwd=WORK, env=os.environ.copy(),
                        start_new_session=os.name == 'posix',
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
def show_progress():
    for line in proc.stdout:
        print(line, end='', flush=True)
progress = threading.Thread(target=show_progress, daemon=True)
progress.start()
hard_timeout = threading.Event()
def stop_all_workers():
    if proc.poll() is None:
        hard_timeout.set()
        # کارگرها session مستقل دارند؛ psutil در Colab/Kaggle معمولاً نصب است.
        # مسیر عادی توسط supervisor داخلی هر worker را به‌صورت مستقل محدود می‌کند.
        try:
            import psutil
            for child in psutil.Process(proc.pid).children(recursive=True):
                try: child.kill()
                except psutil.NoSuchProcess: pass
        except (ImportError, psutil.NoSuchProcess):
            pass
        if os.name == 'posix':
            try: os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError: pass
        else:
            proc.kill()
watchdog = threading.Timer(remaining_seconds + 5, stop_all_workers)
watchdog.daemon = True
watchdog.start()
try:
    returncode = proc.wait()
except KeyboardInterrupt:
    stop_all_workers()
    returncode = proc.wait(timeout=5)
    print('اجرای دستی متوقف شد؛ فایل‌های ساخته‌شده حفظ شده‌اند.')
finally:
    watchdog.cancel()
    progress.join(timeout=2)
print('Exit code:', returncode, '| Elapsed:', round((time.monotonic()-NOTEBOOK_STARTED)/60, 2), 'min')
print('Hard timeout:', hard_timeout.is_set())
print('خروج غیرصفر یعنی یک مرحله ناقص/خطادار بوده؛ جزئیات را در گزارش ببینید.')

## گزارش و معیارها
خطای دبی، معادلات و شرایط مرزی مستقل از loss آموزش گزارش می‌شوند. باقی‌ماندهٔ معادلات درصد خطا در برابر مقاله نیست.

In [ ]:
from IPython.display import display, HTML, Image, FileLink
import pandas as pd
if (OUT / 'summary.json').is_file():
    summary = json.loads((OUT / 'summary.json').read_text())
    print('Workflow complete:', summary['workflow_complete'])
    print('Screening criteria:', summary['research_quality_criteria_met'])
    print('Particles modeled:', summary['particles_modeled'])
    print('Physical time modeled:', summary['physical_time_modeled'])
    print('Literature validation:', summary['physical_validation_cases_run'])
    print('Plot errors:', summary['plot_errors'])
    if (OUT / 'paper_metrics.csv').stat().st_size > 2:
        display(pd.read_csv(OUT / 'paper_metrics.csv'))
    for case in summary['cases']:
        if case['analytic_straight_control'] is not None:
            print('Straight-limit implementation check:', case['analytic_straight_control'])
else:
    print('گزارش نهایی ساخته نشده؛ checkpointها و logها را بررسی کنید:', OUT)

## شکل‌ها و مقایسهٔ حالت‌ها
ساختار شبکه، میدان‌ها، خطوط جریان، دبی، فشار، تنش دیواره و باقی‌مانده‌ها در PNG/PDF/SVG ذخیره می‌شوند. نمای عمق‌دار، حل سه‌بعدی نیست.

In [ ]:
import html
manifest = OUT / 'figure_manifest.json'
if manifest.is_file():
    for figure in json.loads(manifest.read_text()):
        path = OUT / figure['files'][0]
        if path.suffix == '.png' and path.is_file():
            display(HTML('<h4>' + html.escape(figure['title']) + '</h4>'))
            display(Image(filename=str(path), width=900))
            display(HTML('<p>' + html.escape(figure['caption']) + '</p>'))

## اسکن میدان پایا
مقطع مشاهده در طول کانال جابه‌جا می‌شود؛ این نمایش زمان فیزیکی یا حرکت ذره نیست.

In [ ]:
for animation in sorted((OUT / 'cases').glob('*/section_scan.gif')):
    print(animation.parent.name)
    display(Image(filename=str(animation)))

## دریافت خروجی کامل
ZIP شامل شکل‌ها، GIF/MP4، میدان‌های CSV/NPZ، جدول LaTeX، checkpointها، تنظیمات، logها و کد اجرا است.
`index.html` گالری کامل آفلاین و `REPORT.md` گزارش علمی همراه با محدودیت‌ها است.
برای هندسهٔ گرفتگی‌دار، اعتبارسنجی مقاله‌ای ادعا نمی‌شود. اسکن‌ها مربوط به حل پایا هستند.

In [ ]:
archive = OUT.parent / f'{OUT.name}_results.zip'
if archive.is_file():
    display(FileLink(str(archive)))
    print('ZIP:', archive)
    try:
        from google.colab import files
        files.download(str(archive))
    except ImportError:
        print('در Jupyter/Kaggle فایل ZIP را از پنل فایل‌ها دریافت کنید.')
    except Exception as exc:
        print('دانلود خودکار انجام نشد؛ ZIP را از پنل فایل‌ها بگیرید:', exc)
if (OUT / 'completion.json').is_file():
    print(json.dumps(json.loads((OUT / 'completion.json').read_text()), ensure_ascii=False, indent=2))